# A comparison of holography X-ray phase contrast imaging:

# mean intensity vs intensity correlations

X-ray holography with intensity correlations is given by the following forward problem:
$$F(f)=c_{f} ~~\text{with}~~ c_{f}(x,y):=\operatorname{Cov}(I(x),I(y)),~~\text{and}~~I:=|\mathcal{D}e^{f}u|^2 $$
The forward operator is given by:
$$
F(f):=|\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*|^2.
$$
In contrast to the imaging from intensity correlations, the X-ray holography with mean intensity is given by the following forward problem:
$$G(f)=d_{f} ~~\text{with}~~d_{f}(x):=c_{f}(x,x)=\mathbb{E}[I(x)]^2$$
The forward operator is given by:
$$
G(f):=\operatorname{Diag}(\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*).
$$

In [ ]:
import os
import sys

#sys.path.append(os.path.join(os.path.dirname(__file__), '../../'))

from regpy.hilbert import L2
from regpy.vecsps import UniformGridFcts
from regpy.vecsps import NumPyVectorSpace
from regpy.solvers import TikhonovRegularizationSetting
from regpy.solvers.nonlinear.fista import FISTA
import regpy.stoprules as rules
from regpy.operators import FresnelPropagator, SquaredModulus
from regpy.functionals import QuadraticNonneg, QuadraticBilateralConstraints
import matplotlib.pyplot as plt

from auxiliary_ops import ReIm, fresnel_prop
from phaseless_passive_ip_ops import Contrast2FactorPhasedCovOp,  Tau, MatrixAutoProductOp, ExpectationCoxModGaussian, CovarianceCoxModGaussian
from create_Vcov import _create_Vcov
from graphic_utils import test_image_cells, test_image_circle_cross, show_measurements, show_results, show_comparison_results

import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)


# Create forward operators and synthetic data

Setting up parameters 

In [ ]:
N=128                             # Pixel number in each spatial direction
contrast = test_image_circle_cross(N,N)
N_frame=3000                      # Number of frames ( or realizations)
T=1e9 / N_frame                   # the observation time or the number of photon counts
fresnel_number=10                 # Fresnel number
#fresnel_number = 10
N_b=4                             # denotes the rank of the matrix V

construct decomposition $\operatorname{Cov}[u]=VV^*$ of the source covariance operator 

In [ ]:
create_types=['spatial', 'Fresnelprop', 'fourier_random']
create_type='fourier_random'
sigma=0.5                         # parameter used in the rapid deacaying function

grid=UniformGridFcts((-1, 1, N), (-1, 1, N), dtype=complex,periodic=True)
Vcov, S=_create_Vcov(N, N_b, create_type=create_type, grid=grid, sigma=sigma, xsample=grid.axes[0], ysample=grid.axes[1])
# here S is the singular values of the matrix V
Vcov = Vcov * (1./S)[(None,None,...)]

compute forward operators 

In [ ]:
fp = FresnelPropagator(grid,fresnel_number,pad_amount=200)
Mat_op=Contrast2FactorPhasedCovOp(Vcov,fp)
ReIm_op=ReIm(grid)

# forward operator associated with the intensity correlations
Cov_Cox = CovarianceCoxModGaussian(Mat_op.codomain,fp.codomain,T)
op_intcorr = Cov_Cox * Mat_op * ReIm_op.adjoint

# forward operator associated with the mean intensity
E_Cox = ExpectationCoxModGaussian(Mat_op.codomain,fp.codomain)
op_meanint = E_Cox * Mat_op * ReIm_op.adjoint

In [ ]:
im = fp(contrast)
fp2 = FresnelPropagator(grid,fresnel_number,pad_amount=100)
im2 = fp2(contrast)
fp3 = FresnelPropagator(grid,fresnel_number,pad_amount=200)
im3 = fp3(contrast)

fig, (ax1,ax2) = plt.subplots(1,2)
ax1.imshow(np.abs(im))
ax1.set_title('without zero padding')
ax2.imshow(np.abs(im3))
ax2.set_title('with zero padding')


Create synthetic intensity data 

In [ ]:
ptw_detection= SquaredModulus(grid)
taumat_0=Mat_op(contrast) 
data_meanint=np.zeros((N, N))   # mean intensity 
intensities=np.zeros((N_frame, N, N))

for i in range(0, N_frame):
    random=1/np.sqrt(2)*(np.random.randn(N_b)+complex(0,1)*np.random.randn(N_b))
    uinc=np.tensordot(taumat_0, random, axes=([-1], [0]))
    signal=ptw_detection(uinc)
    
    #Cox-process
    signal=(1/T)*np.random.poisson(lam=T*signal.flatten(), size=(N**2)).reshape(N, N)
    data_meanint += signal
    intensities[i, :, :]=signal
    
data_meanint /= N_frame

# centering of  intensities
intensities -= data_meanint

In [ ]:
show_measurements(Vcov,intensities,data_meanint)

# Inversions
Both for mean intensity data and for intensity correlations we use Tikhonov regularization with L^2 data fidelity term and an L^2 penalty term 
incorporating a non-negativity constraint. The regularization parameter is decreased gradually, using the minimizer of the previous Tikhonov functional as initial guess for the next one. Minimizers of the Tikhonov functional are computed by the FISTA algorithm.

## Inversion for mean intensity

initialize inversions

In [ ]:
regpar_meanint = 1e-8    # initial regularization parameter
Nfista_meanint = 10      # number of FISTA steps for each value of the regularization parameter

penalty=QuadraticNonneg(ReIm_op.codomain)

setting_mean_inten = TikhonovRegularizationSetting(op=op_meanint, penalty=penalty, data_fid = L2, 
                                        data_fid_shift=data_meanint,regpar=regpar_meanint)

reco_meanint = np.zeros_like(contrast)   # initial guess is zero
mean_int_stats = []                      # reconstructions and errors for each regularization parameter will stored   

iteratively decrease the regularization parameter

In [ ]:
for _ in range(3):
    setting_mean_inten.regpar = setting_mean_inten.regpar/3.
    FISTA_solver = FISTA(setting_mean_inten,init = ReIm_op(reco_meanint),without_codomain_vectors=False,logging_level="INFO")
    stoprule = rules.CountIterations(Nfista_meanint,logging_level="WARN")
    reco_meanint, reco_data_meanint=FISTA_solver.run(stoprule)
    reco_meanint=ReIm_op.adjoint(reco_meanint)

    error_meanint=np.linalg.norm(reco_meanint-contrast)/np.linalg.norm(contrast)
    print(f"error_meanint = {error_meanint}, alpha= {setting_mean_inten.regpar}")
    mean_int_stats.append([reco_meanint,error_meanint,setting_mean_inten.regpar])
show_results(contrast,reco_meanint)

In [ ]:
for _ in range(1):
    setting_mean_inten.regpar = setting_mean_inten.regpar
    FISTA_solver = FISTA(setting_mean_inten,init = ReIm_op(reco_meanint),without_codomain_vectors=False,logging_level="INFO")
    stoprule = rules.CountIterations(Nfista_meanint,logging_level="WARN")
    reco_meanint, reco_data_meanint=FISTA_solver.run(stoprule)
    reco_meanint=ReIm_op.adjoint(reco_meanint)

    error_meanint=np.linalg.norm(reco_meanint-contrast)/np.linalg.norm(contrast)
    print(f"error_meanint = {error_meanint}, alpha= {setting_mean_inten.regpar}")
    mean_int_stats.append([reco_meanint,error_meanint,setting_mean_inten.regpar])
show_results(contrast,reco_meanint)

In [ ]:
[(err[1],err[2]) for err in mean_int_stats]

## Inversion for intensity correlations

initialize inversions

In [ ]:
regpar_intcorr = 1e-7     # initial regularization parameter
Nfista_intcorr = 10       # number of FISTA steps for each value of the regularization parameter

penalty=QuadraticNonneg(ReIm_op.codomain)

#reco_intcorr = None # use this for ab initio reconstruction based on intensity correlations
reco_intcorr = reco_meanint # use this for a warm start from mean intensity reconstruction

setting_intcorr = TikhonovRegularizationSetting(op=op_intcorr, penalty=penalty, data_fid = L2,
                                            regpar=regpar_intcorr)

intcorr_stats = []

iteratively decrease the regularization parameter

In [ ]:
for _ in range(2):
    setting_intcorr.regpar = setting_intcorr.regpar/3
    FISTA_solver = FISTA(setting_intcorr,data=intensities,init = ReIm_op(reco_intcorr),without_codomain_vectors=True)
    stoprule = (rules.CountIterations(2*Nfista_intcorr))
    reco_intcorr, reco_data_intcorr=FISTA_solver.run(stoprule)
    reco_intcorr=ReIm_op.adjoint(reco_intcorr)

    error_intcorr=np.linalg.norm(reco_intcorr-contrast)/np.linalg.norm(contrast)
    print(f"error_intcorr = {error_intcorr}, alpha= {setting_intcorr.regpar}",)
    intcorr_stats.append([reco_intcorr,error_intcorr,setting_intcorr.regpar])
show_results(contrast,reco_intcorr)


In [ ]:
[(err[1],err[2]) for err in intcorr_stats]

## Comparative plots

In [ ]:
show_comparison_results(contrast,reco_intcorr,reco_meanint)

